This lesson introduces fundamental optimization concepts which will be needed for machine learning. We will start with intuitions around trial and improvement, then formalize this to optimization and introduce the concepts of objective functions and implement the random search algorithm.

### Imports and seeding

In [1]:
import random as r
import numpy as np
r.seed(1)

# Trial and Improvement

Trial and improvement is the intuitive way that humans guess at solutions to a problem. For example, suppose we do not know the value of $\pi$ and can't just use np.pi, how do we find the value of pi? An obvious approach is to have a criterion which holds for $\pi$ then keep guessing values for $\pi$ until that criterion holds. 

Suppose we know $\pi$ is more than 1 and less than 5. We also know that $sin(\pi) = 0$, so we'll guess values $x$ of pi until we find a value such that $|sin(x) < \epsilon|$ for some small threshold $\epsilon$ which we choose, then we stop and output our value for $x$.

In [2]:
threshold = 10 ** -3

def generate_hypothesis():
    return(r.uniform(1, 5))

x = generate_hypothesis()
while np.fabs(np.sin(x)) > threshold:
    x = generate_hypothesis()
print(f"Trial and improvement says pi ~{x}")

Trial and improvement says pi ~3.142476023145567


That worked! We got $\pi \approx 3.14$ fairly quickly without having to do anything too complex, we just kept guessing until a guess worked. Let's formalize what we just did. 

We had a hypothesis generator $G$ which takes no input and generates a hypothesis $x$ for an answer to our problem and a criterion $C(x)$ which takes our $x$ and outputs true if the criterion holds and false otherwise. We repeatedly set $x = G(x)$ and stopped when $C(x)$ is true.

This worked well so let's extend it to the random search algorithm.

# Random Search

Trial and Improvement worked because we knew the search domain was small ($400$ hypothesess to 2.d.p of $\pi$) but often the search domain is very large and we cannot be sure we will find an ideal solution where $C(x)$ holds. If trial and improvement doesn't find a solution where $C(x)$ is true within an acceptable runtime then we just spent a whole bunch of compute and achieved nothing! 

Instead, let's set a budget of how many guesses we are allowed to make $N$ and come up with a procedure which makes progress even if $C(x)$ never holds, for now set $N = 10^4$. We'll define an "objective function" $L(x)$ which tells us how well we are doing. If we are trying to increase our function it is called a "reward function" or a "utility function", and if we are trying to decrease it we call it a "loss function". We define:

$L(x) = |sin(x)|$

And minimize $L(x)$. Here, $L(x)$ is effectively a way of measuring how wrong our answer is. To minimize $L(x)$, we have our current guess $x^*$ and while we are below $N$ we will set $x = G(x)$. If $L(x) < L(x^*)$ then our new $x^*$ is $x$.

In [3]:
def loss(x):
    return(np.fabs(np.sin(x)))

num_iterations = 10 ** 4
x_star = generate_hypothesis()
y_star = loss(x_star)
for i in range(num_iterations):
    x_new = generate_hypothesis()
    y_new = loss(x_new)
    if y_new < y_star:
        y_star = y_new
        x_star = x_new
print(f"Random search says pi ~ {x_star}")

Random search says pi ~ 3.1414533046415154


# Harder Problems

Guessing values for $\pi$ is easy, so let's look at a more interesting problem. Suppose we work for a company which manufactures soft drinks. We want to come up with a design for a can such that we hold a required volume of pop, say 330 ml, but otherwise use as little metal as possible. Suppose also that the can is approximately a cylinder.

Recall that for a cylinder $V(r, h) = \pi r^2 h$ but since $V = 330cm^3$ we can set $h = \frac{330}{\pi r^2}$  (recall that $1ml = 1cm^3$).

Then the surface area of the cylinder can therefore be written only in terms of $r$ as follows:

$A(r) = 2\pi r^2 + 2\pi r h$ (two circles plus the curved rectangle around them)

$= 2\pi(r^2 + h)$

$= 2\pi(r^2 + \frac{330}{\pi r^2})$

So we have $A(r)= 2\pi(r^2 + \frac{330}{\pi r^2})$ but notice that the constants outside of the bracket don't matter: if we minimize a constant times a function, we also minimize that function. So let's set:-

$L(x) = (x^2 + \frac{330}{\pi x^2})$

and minimize $L(x)$. We'll update the hypothesis generator and loss function, but otherwise our code is the same. Also it's a good idea to look up an expected result in advance, so according to a quick search a typical 330ml can has a diameter of about 66mm. We are finding a radius in cm, so we expect an answer around 3.3.

In [4]:
#Consider radii from 0cm to 8cm
def generate_hypothesis():
    return(r.uniform(0, 8))
    
def loss(x):
    return(x ** 2 + (330 / (np.pi * x ** 2)))

num_iterations = 10 ** 4
x_star = generate_hypothesis()
y_star = loss(x_star)
for i in range(num_iterations):
    x_new = generate_hypothesis()
    y_new = loss(x_new)
    if y_new < y_star:
        y_star = y_new
        x_star = x_new
print(f"Random search produces radius: {x_star:.2f}cm")

Random search found the radius: 3.20cm


So that's a good result, we are within 1mm of the stated radius for real cans. 

# Task - Optimal Speed

The resistance a vehicle recieves per second given it is travelling at speed $v$ is given by:

$R(v) = a + bv + cv^2$

Using the random search algorithm, find the optimal speed for a car with $a = 200 N$, $b = 0.01 Nms^{-1}$, and $c = 0.3Nms^{-2}$ to travel at in order to minimize the total resistance it experiences along a journey of constant length. Convert your answer to mph or kph to make sure it makes sense!

Hints:

- do not minimize $R(v)$, you will need to think about it and minimize some other, related function.
- the distance the vehicle travels does not matter
- use random search
- the answer is not $0ms^{-1}$
- be careful with units! We are working in $ms^{-1}$, not mph or kph.